# Process All 255 Videos — EMNLP StandUp4AI

**Goal:** Extract 791-dim features for all 255 videos with audio+EMNLP labels.
Train FusionMLP, evaluate on held-out test set with IoU metrics.

**Dataset:** 255 videos from StandUp4AI en_uk partition with:
- Word-level timestamps from EMNLP CSV labels
- BIO tags: O (non-laugh), B/I/L (laugh)
- Audio from gdrive:standup4ai/audio/ or audio_1000/

**Runtime:** GPU (T4 or A100) | **Time:** ~4-6 hours

In [ ]:
# Cell 0: Setup
import os, sys, json, subprocess, warnings
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Install dependencies
subprocess.run([sys.executable, '-m', 'pip', 'install', 
    'yt-dlp', 'soundfile', 'librosa', 'transformers', 'torch', '-q'],
    capture_output=True)

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
BASE = '/content/drive/MyDrive/standup4ai'
WORK = '/content/process255'
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{WORK}/features', exist_ok=True)
os.makedirs(f'{WORK}/audio', exist_ok=True)
print(f'WORK: {WORK}')
print(f'BASE: {BASE}')


In [ ]:
# Cell 1: 255 Video IDs with Audio + EMNLP Labels
# Verified overlap between audio folders and en_uk labels
ALL_VIDEOS = [
    '-UPIA46hBZs', '-vcKXr6WBNc', '0AvUvJ_S2Os', '0Pl51hxcK-o',
    '0g7nezWZyfY', '0zpUnJSG0EQ', '18H1aeoGybw', '18rLwnvxOU0',
    '1ILQmgHvtd4', '1Uo27tH3JQ4', '1pPnJut3KLw', '1tO9MWWOgHk',
    '1u-pq9LLWlU', '21gOjz-Xk7s', '2SUfHIbT0HI', '2axWotdMFsw',
    '2ql8QJWmNM8', '3TgRGK1vrzs', '3ZTClwMxpmM', '3new05S61w4',
    '41piF6uPhXg', '482LeT9UT7I', '4ZiXvhSxnD4', '53JXuJGmhoU',
    '5bKcTy3zag4', '5cdoHY0ziVA', '5gp79fSWHy0', '66CyaeFWucM',
    '6Ofc2A75zuw', '76r8IcowEsE', '7E7la6BCpRc', '7Gw1NjZ13fA',
    '7VkAFkK3bwQ', '7cBFWZDXlHA', '7gRo0nF1yS0', '7kULz2NevT4',
    '8CoHAczz9pY', '8EUpV_qyEpc', '8eYSNXOsyoo', '8nltoWdciws',
    '90s9HfZhM0Y', '9DwiBEVDdUE', '9h7-OMYItDI', '9yPco6WNYG0',
    'AES4jzE513Y', 'AEnlxaPVtK8', 'AI69HZWZ26c', 'A_EIL1ojfK4',
    'Azl5GJuYqE0', 'B9jLEExvazc', 'BT-WOZQ5JRc', 'BbBymwvs7co',
    'Bl-PyS4f8as', 'BoMFeYyvYP8', 'C1TMo0YTDLA', 'CNKnRGig1FM',
    'CUEvqRwSi_c', 'CgeJOMi1mEU', 'CocEMvDXiu8', 'CwMCoAh1-a0',
    'CwYov1S2060', 'DN-dSUoKuJQ', 'DN-vrVTIKwk', 'DcwbxzMMPEM',
    'DlS85HEFvlE', 'DsSh_4VSXP4', 'E8ToU_gqdlY', 'EIpbdW9Vb6s',
    'EPY7mNtnKy8', 'EtdeJ-2bYO0', 'Eva3iridbd4', 'FBX2tyYrjog',
    'FDKzqoSjFxs', 'FUbXP88a0wk', 'FcOGp4y2O3I', 'G-BPYtfjpS0',
    'GCfX2_fRSIk', 'G_hDEQDtEf0', 'Ga1dQxpUft0', 'H3Y-9-CarcQ',
    'HIooOHs_8sg', 'HKJKaIwCCQk', 'HjTy45N6G6E', 'HtPGDsZEIC0',
    'IFtWoIw0DVE', 'IIhav6q5IsE', 'INhj_TbMRXk', 'J9HLFJgUCW0',
    'JIWQBC8Q1e8', 'JLOjHhWTKLA', 'JMbjs8fLyaY', 'JaRBJnElNZc',
    'JsL-FKqlHD0', 'Jw_jJ8fE9ZA', 'KRMi6FzC2vM', 'KfSI5Krer8g',
    'KicH8MlWxn4', 'KqpzX1oH-E0', 'Ku-mGC3pRsk', 'LH6JJn5PPWg',
    'LYXsdcLkIVY', 'M1NDZYLSo94', 'M5Utz5IROls', 'MA6B8NQ9oxo',
    'MBmldf0UW-A', 'MHLcCWGpyRc', 'N7Z7Xc3sJuE', 'NurQtpa590E',
    'NwKrc00JPVA', 'OESjhjRYBUQ', 'OZnoJ9WoLGw', 'O_YGnqj_z2M',
    'Obc1d4v4x2U', 'P0-fCn2ptvc', 'PQyFD9DRKpk', 'PVaNYuVeRE0',
    'Q1EytxIpFIo', 'QJ6tmMFp0Rs', 'QO8QC0QpPr4', 'QPWJh6Jde7c',
    'QRC1wN7dwPM', 'R5HTUMhnabA', 'RTEjVNUKsXE', 'RuciD4LJo6Q',
    'S18ykYyYEr4', 'S2SXqVjmJ9E', 'S5oYYq2KPoc', 'Sfv6oiA89O8',
    'Sl8Zazk2C3s', 'SofFDLTpy68', 'TdAMvTod6zQ', 'UI2Bb1zAFw4',
    'UPgE_HxS88Q', 'Ucz9sCT6ydQ', 'Utog2sKMYOI', 'VLyPoVtMxzg',
    'VSytmEkhOdA', 'Vle0vKahRzA', 'VxNJxDKkLGo', 'W3jHBME1C0M',
    'WN9u1jSMRGM', 'WrklYFQIYRI', 'WtfN5loZa08', 'XKGDf_btalc',
    'XMwZnO7teGA', 'X_MsBNNePpw', 'YAqJM9QFmgo', 'YZzO06XQAw0',
    'Z1ChLkF6ooE', 'ZV8Xgf0i1gE', 'ZpF_bEUkqnk', 'ZxsuaSEKDyk',
    '_0g4b38_Kdk', '_BY8M7cRVwQ', '_IYp2Sq2Yng', '_QdTi-N_Pgk',
    'a7WbsSPVQwQ', 'aU7VUkOLx6s', 'apSIp7FvEwI', 'aqnqgMxfpFo',
    'ayMuQVc8DWA', 'bOYGQFO65c0', 'bUf7sYCYI6A', 'bXKT1OedpZk',
    'cOTolCho3r8', 'cObKxofJ1ww', 'cl2qN3mbfXs', 'e5p8aXTgFgo',
    'em6rlnYLspg', 'f-eadBMRiMw', 'fQjbi5VRNs4', 'gOniIHA6xqU',
    'gVw7B-EXNfo', 'gXjE2gqcs0U', 'hRRvWVTHK8k', 'hqAuogvh8Rk',
    'i8VjBdvQ0Ko', 'i9UsxfvOzSs', 'iym4rS5WT8U', 'izYxUn2WXfc',
    'jF6Devdvzqo', 'joBWJJ2b457o', 'k1Tl-uNpey4', 'kMiPVkw8dlQ',
    'kMkoV4xoadQ', 'kjH9M7Mkwzk', 'kwV4zErO5jw', 'l2oaxKORheA',
    'lG-W3DNL4Ps', 'lGjXQbZ_GAM', 'lNi9kBtraPE', 'lZJinxkxvOc',
    'l_UVH2OJOIc', 'lgySVdjX86E', 'lnd9QY-Sa20', 'lzKSw7679PU',
    'm8X0F6diPDg', 'mA_yzRl5bfc', 'mG0SEr6jZSo', 'ml-wZ-Os-04',
    'n3aBK6vd4rM', 'nJK5wIkM7V0', 'nQpaY1-_LMY', 'nlol_hJwEr8',
    'o2U9y8rBLL0', 'oGWVsPp4a5E', 'pV_KOZ1jJTU', 'pdyGbfrMyLo',
    'pnBa8uvzqtQ', 'q112mLKiUCw', 'qCilT_NrKac', 'qyFw-BnDyes',
    'rAPZ26R8on8', 'rHfWihSHAGQ', 'rK_FOkWcwG4', 'rQKN_sp00O8',
    'rRYPyKo_1l8', 'rRYa8Cd8NSE', 'rjiH4VcYrnA', 'sVxPD081Xa8',
    'sXn99zQfges', 'spZC_hrHTe0', 'tG2qlpdNPbo', 'tJKiK1WcA8s',
    't_1ULTlaZ4I', 'u7eWgKgRau8', 'uORo9BRI1EQ', 'v7USDAkEzdE',
    'vc37gyX0F7g', 'viSnHyBxYAA', 'vn9i61wzYLQ', 'vtVfYS3c1RU',
    'vuJfIw425f8', 'w3AzeGKnNAY', 'wM1BECRNygc', 'wReG9acljeU',
    'wUSiztgPM8A', 'wd4K5OJU_BU', 'wwCxu2yJfSU', 'x1iBps71YPs',
    'x9T-oDFcb1I', 'xPoyN5wzapQ', 'xXi_AAgaZOo', 'xh06K8YYyN8',
    'xlbl--hurDQ', 'xvNoe0HZnVc', 'y-9nbIdUpbA', 'yGgqulbbxgA',
    'yYAjshQA2ms', 'yYdkqNt4azU', 'z2NcUVLwH-Y', 'z4Wht3FpzPg',
    'zEcWVrCAk0A', 'zNOZzpm3fz8', 'zfMg7k1swX8'
]

# Split into train/val/test by video ID hash (deterministic)
import hashlib
def hashvid(v):
    return int(hashlib.md5(v.encode()).hexdigest(), 16)

train_vids, val_vids, test_vids = [], [], []
for v in sorted(ALL_VIDEOS, key=hashvid):
    h = hashvid(v) % 100
    if h < 70:
        train_vids.append(v)
    elif h < 85:
        val_vids.append(v)
    else:
        test_vids.append(v)

print(f'Train: {len(train_vids)}, Val: {len(val_vids)}, Test: {len(test_vids)}')
print(f'Total: {len(train_vids) + len(val_vids) + len(test_vids)}')

# Save split
with open(f'{WORK}/split.json', 'w') as f:
    json.dump({'train': train_vids, 'val': val_vids, 'test': test_vids}, f, indent=2)
print(f'Saved: {WORK}/split.json')


In [ ]:
# Cell 2: Load Models (GPU)
import torch, torch.nn as nn
from transformers import AutoModel
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
assert str(device) == 'cuda', 'GPU required for WavLM'

# Load WavLM
print('Loading WavLM...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print(f'WavLM ready: {sum(p.numel() for p in wavlm.parameters()):,} params')

# Constants
SR_WAVLM, SR_PROSODY = 16000, 22050

# Prosody extractor (matches 23-dim training data)
def prosody23(y, sr):
    f = []
    try:
        f0, vd, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0c = f0[~np.isnan(f0)]; v = vd[~np.isnan(f0)]
        f.extend([np.mean(f0c) if len(f0c)>0 else 0,
                  np.std(f0c) if len(f0c)>0 else 0,
                  np.max(f0c) if len(f0c)>0 else 0,
                  np.min(f0c) if len(f0c)>0 else 0,
                  np.mean(v) if len(v)>0 else 0])
    except: f.extend([0]*5)
    hop = 512
    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    f.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms)-np.min(rms)])
    dur = len(y)/sr
    f.extend([dur, dur/(np.sum(rms>np.mean(rms))+1)])
    try:
        sc = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
        sb = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop)[0]
        sf2 = librosa.feature.spectral_flatness(y=y, hop_length=hop)[0]
        zcr = librosa.feature.zero_crossing_rate(y, hop_length=hop)[0]
        f.extend([np.mean(sc), np.mean(sb), np.mean(sf2), np.mean(zcr), np.std(zcr)])
    except: f.extend([0]*5)
    try:
        yh, _ = librosa.effects.hpss(y)
        hnr = np.mean(np.abs(yh))/(np.mean(np.abs(y))+1e-8)
        f.extend([hnr, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y)), 0, 0])
    except: f.extend([0]*6)
    return np.array(f[:23], dtype=np.float32)

def word_features(y16, y22, t0, t1):
    """Extract 791-dim for one word [t0, t1] seconds."""
    dur = t1 - t0
    if dur < 0.02: return None
    s16, e16 = int(t0*SR_WAVLM), min(int(t1*SR_WAVLM), len(y16))
    c16 = y16[s16:e16]
    if len(c16) < 0.1*SR_WAVLM: return None
    if len(c16) < 5*SR_WAVLM:
        c16 = np.pad(c16, (0, int(5*SR_WAVLM)-len(c16)))
    with torch.no_grad():
        wemb = wavlm(torch.tensor(c16/32768.0).unsqueeze(0).to(device)).last_hidden_state.mean(1).squeeze().cpu().numpy()
    s22, e22 = int(t0*SR_PROSODY), min(int(t1*SR_PROSODY), len(y22))
    pros = prosody23(y22[s22:e22], SR_PROSODY)
    return np.concatenate([wemb, pros])  # (791,)

print('Feature extractor ready: WavLM(768) + prosody(23) = 791')


In [ ]:
# Cell 3: Check which videos we already have features for (checkpoint resume)
CKPT_FILE = f'{WORK}/extract_ckpt.json'
DONE_FILE = f'{WORK}/features_done.json'

done = set()
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f:
        done = set(json.load(f))
    print(f'Resuming: {len(done)} videos already processed')

remaining = [v for v in train_vids + val_vids + test_vids if v not in done]
print(f'Remaining to process: {len(remaining)}/{len(ALL_VIDEOS)}')


In [ ]:
# Cell 4: Download Audio for a Batch
# Try Drive first, then yt-dlp as fallback
from tqdm import tqdm
import shutil

def get_audio(vid):
    """Get audio path. Tries Drive then yt-dlp."""
    # Try Drive locations
    for folder in [f'{BASE}/audio', f'{BASE}/audio_1000']:
        p = f'{folder}/{vid}.m4a'
        if os.path.exists(p):
            dst = f'{WORK}/audio/{vid}.m4a'
            if not os.path.exists(dst):
                shutil.copy(p, dst)
            return dst
    
    # Try local audio
    local = f'{WORK}/audio/{vid}.m4a'
    if os.path.exists(local):
        return local
    
    # Fallback: yt-dlp
    out = f'{WORK}/audio/{vid}.wav'
    if os.path.exists(out): return out
    cmd = ['yt-dlp', '-f', 'bestaudio[ext=m4a]', '-o', f'{out}.%(ext)s',
            f'https://www.youtube.com/watch?v={vid}',
            '--no-playlist', '--quiet', '--socket-timeout', '90',
            '--extract-audio', '--audio-format', 'wav']
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=180)
    if r.returncode == 0:
        for ext in ['m4a', 'webm', 'mp4']:
            tmp = f'{out}.{ext}'
            if os.path.exists(tmp):
                os.rename(tmp, out)
        if os.path.exists(out):
            return out
    return None

# Download first 50 remaining (batch)
batch = remaining[:50]
print(f'Downloading batch of {len(batch)}...')
for i, vid in enumerate(tqdm(batch)):
    path = get_audio(vid)
    status = 'OK' if path else 'FAIL'
    print(f'  [{i+1}/{len(batch)}] {vid} {status}')

n_audio = len([f for f in os.listdir(f'{WORK}/audio') 
                if f.endswith(('.m4a', '.wav'))])
print(f'Total audio files: {n_audio}')


In [ ]:
# Cell 5: Extract Features for All Videos
# Process all 255 videos with available audio
import pandas as pd
from tqdm import tqdm

# Get audio files
audio_files = {}
for f in os.listdir(f'{WORK}/audio'):
    if f.endswith(('.m4a', '.wav')):
        vid = f.rsplit('.', 1)[0]
        audio_files[vid] = f'{WORK}/audio/{f}'

print(f'Audio files available: {len(audio_files)}')

# Load checkpoint
done = set()
if os.path.exists(DONE_FILE):
    with open(DONE_FILE) as f:
        done = set(json.load(f))

processed = 0
errors = []

for vid in tqdm(ALL_VIDEOS):
    if vid in done:
        continue
    
    # Get audio
    if vid not in audio_files:
        continue  # No audio, skip
    
    # Get labels
    label_paths = [
        f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train/{vid}.csv',
        f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/val/{vid}.csv',
    ]
    label_path = None
    for lp in label_paths:
        if os.path.exists(lp):
            label_path = lp
            break
    
    if not label_path:
        continue  # No labels, skip
    
    try:
        # Load audio at both sample rates
        y22, _ = librosa.load(audio_files[vid], sr=SR_PROSODY, mono=True)
        y16, _ = librosa.load(audio_files[vid], sr=SR_WAVLM, mono=True)
        
        # Load labels
        df = pd.read_csv(label_path)
        
        # Extract features per word
        features_list = []
        labels_list = []
        
        for _, row in df.iterrows():
            try:
                ts = eval(str(row['timestamp']))
                t0, t1 = float(ts[0]), float(ts[1])
                feat = word_features(y16, y22, t0, t1)
                lbl = str(row.get('label', 'O')).strip()
                
                if feat is not None:
                    # Binary label: 0=non-laugh, 1=laugh
                    is_laugh = 1 if lbl in ('B', 'I', 'L') else 0
                    features_list.append(feat)
                    labels_list.append(is_laugh)
            except:
                pass
        
        if features_list:
            features = np.array(features_list, dtype=np.float32)  # (n_words, 791)
            labels = np.array(labels_list, dtype=np.int32)         # (n_words,)
            
            np.save(f'{WORK}/features/{vid}_features.npy', features)
            np.save(f'{WORK}/features/{vid}_labels.npy', labels)
            
            done.add(vid)
            processed += 1
            
            # Save checkpoint every 20 videos
            if len(done) % 20 == 0:
                with open(DONE_FILE, 'w') as f:
                    json.dump(sorted(done), f)
                print(f'\nCheckpoint: {len(done)} videos done')
        
    except Exception as e:
        errors.append((vid, str(e)))

# Final checkpoint
with open(DONE_FILE, 'w') as f:
    json.dump(sorted(done), f)

print(f'\nProcessed: {processed} videos')
print(f'Total done: {len(done)}/{len(ALL_VIDEOS)}')
if errors:
    print(f'Errors: {len(errors)}')
    for vid, err in errors[:5]:
        print(f'  {vid}: {err}')


In [ ]:
# Cell 6: Train FusionMLP with 5-Fold GroupKFold
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score

class FusionMLP(nn.Module):
    def __init__(self, input_dim=791):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), nn.ReLU(), nn.BatchNorm1d(512), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.3),
            nn.Linear(256, 64), nn.ReLU(), nn.BatchNorm1d(64), nn.Dropout(0.3),
            nn.Linear(64, 1))
    def forward(self, x):
        return self.net(x)

# Load all features
feat_dir = f'{WORK}/features'
X_list, y_list, vids_list = [], [], []

for vid in train_vids + val_vids:
    fp = f'{feat_dir}/{vid}_features.npy'
    lp = f'{feat_dir}/{vid}_labels.npy'
    if os.path.exists(fp) and os.path.exists(lp):
        X_list.append(np.load(fp))
        y_list.append(np.load(lp))
        vids_list.extend([vid] * len(np.load(lp)))

X_all = np.vstack(X_list)
y_all = np.concatenate(y_list)
groups = np.array(vids_list)

pos_rate = y_all.mean()
print(f'Data: {len(y_all)} words, {y_all.sum()} pos ({100*pos_rate:.1f}%)')
print(f'Unique videos: {len(set(vids_list))}')

# Auto-compute pos_weight (capped at 3.0)
pos_weight = min((1.0 - pos_rate) / max(pos_rate, 1e-6), 3.0)
print(f'pos_weight: {pos_weight:.2f}')

# 5-fold GroupKFold
gkf = GroupKFold(n_splits=5)
models, scalers, fold_f1s = [], [], []

for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_all, y_all, groups)):
    print(f'\n=== Fold {fold+1}/5 ===')
    Xtr, Xte = X_all[tr_idx], X_all[te_idx]
    ytr, yte = y_all[tr_idx], y_all[te_idx]
    
    # Check saturation
    print(f'  Train: {len(ytr)} words, Val: {len(yte)} words')
    
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    Xte_s = scaler.transform(Xte)
    
    model = FusionMLP()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight]).to(device))
    
    Xtr_t = torch.tensor(Xtr_s, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1).to(device)
    
    best_f1, patience, no_improve = 0, 5, 0
    for epoch in range(50):
        model.train()
        for i in range(0, len(Xtr_t), 256):
            bx = Xtr_t[i:i+256]
            by = ytr_t[i:i+256]
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()
        
        model.eval()
        with torch.no_grad():
            probs = torch.sigmoid(
                model(torch.tensor(Xte_s, dtype=torch.float32).to(device)
            )).cpu().numpy().squeeze()
            f = f1_score(yte, (probs >= 0.5).astype(int))
        
        if f > best_f1:
            best_f1 = f
            no_improve = 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    
    # Final eval
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(
            model(torch.tensor(Xte_s, dtype=torch.float32).to(device))
        ).cpu().numpy().squeeze()
        
        # Saturation check
        if probs.std() < 0.01:
            print(f'  WARNING: Model saturated! prob_std={probs.std():.6f}')
        
        p = precision_score(yte, (probs >= 0.5).astype(int), zero_division=0)
        r = recall_score(yte, (probs >= 0.5).astype(int), zero_division=0)
        f = f1_score(yte, (probs >= 0.5).astype(int), zero_division=0)
        print(f'  F1={f:.4f} P={p:.4f} R={r:.4f}')
    
    models.append(model.cpu())
    scalers.append(scaler)
    fold_f1s.append(f)

print(f'\n=== CV F1: {np.mean(fold_f1s):.4f} +/- {np.std(fold_f1s):.4f} ===')
print(f'Fold F1s: {[round(f, 4) for f in fold_f1s]}')


In [ ]:
# Cell 7: Evaluate on Test Set (IoU Metrics)
# IoU-based segment-level evaluation against EMNLP ground truth

def bio_to_laugh_segments(df):
    """Convert word-level BIO to (start, end) laugh segments."""
    spans, i = [], 0
    while i < len(df):
        lbl = str(df.iloc[i].get('label','')).strip()
        ts = eval(str(df.iloc[i]['timestamp']))
        if lbl == 'L':
            spans.append((float(ts[0]), float(ts[1])))
        elif lbl == 'B':
            st, en = float(ts[0]), float(ts[1])
            j = i + 1
            while j < len(df):
                nl = str(df.iloc[j].get('label','')).strip()
                if nl in ('I', 'L'):
                    en = float(eval(str(df.iloc[j]['timestamp']))[1])
                    j += 1
                else:
                    break
            spans.append((st, en))
            i = j - 1
        i += 1
    return spans

def span_iou(s1, s2):
    inter = max(0.0, min(s1[1],s2[1]) - max(s1[0],s2[0]))
    union = max(s1[1],s2[1]) - min(s1[0],s2[0])
    return inter/union if union > 0 else 0.0

def segment_f1(pred, gt, th=0.3):
    if not pred or not gt: return 0.0, 0.0, 0.0
    mp, mg = set(), set()
    for pi, ps in enumerate(pred):
        bi, bg = 0.0, -1
        for gi, gs in enumerate(gt):
            if gi in mg: continue
            iv = span_iou(ps, gs)
            if iv >= th and iv > bi: bi, bg = iv, gi
        if bg >= 0: mp.add(pi); mg.add(bg)
    tp = len(mp)
    p = tp/len(pred) if pred else 0.0
    r = tp/len(gt) if gt else 0.0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0.0
    return p, r, f

def merge_probs(probs, timestamps, thr=0.5):
    spans, in_seg, start = [], False, 0.0
    for i, (pr, (t0, t1)) in enumerate(zip(probs, timestamps)):
        if pr >= thr and not in_seg: in_seg, start = True, t0
        elif pr < thr and in_seg: in_seg = False; spans.append((start, t0))
    if in_seg: spans.append((start, timestamps[-1][1]))
    return spans

# Evaluate on test videos
THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]
RESULTS = {th: [] for th in THRESHOLDS}
per_video = []

for vid in tqdm(test_vids):
    fp = f'{WORK}/features/{vid}_features.npy'
    lp = f'{WORK}/features/{vid}_labels.npy'
    if not os.path.exists(fp) or not os.path.exists(lp):
        continue
    
    # Load features
    features = np.load(fp)  # (n_words, 791)
    
    # Get ground truth from labels
    labels = np.load(lp)  # (n_words,)
    
    # Get timestamps from CSV
    label_paths = [
        f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train/{vid}.csv',
        f'{BASE}/seq-Standup4AI/dataset/en_uk/emnlp+jahak/val/{vid}.csv',
        f'{BASE}/seq-Standup4AI/dataset/en_uk/Manual/test/{vid}.csv',
    ]
    label_path = next((p for p in label_paths if os.path.exists(p)), None)
    if not label_path: continue
    
    df = pd.read_csv(label_path)
    timestamps = []
    valid_mask = []
    for _, row in df.iterrows():
        try:
            ts = eval(str(row['timestamp']))
            timestamps.append((float(ts[0]), float(ts[1])))
            lbl = str(row.get('label', 'O')).strip()
            valid_mask.append(1 if lbl in ('B', 'I', 'L', 'O') else 0)
        except:
            pass
    
    if len(timestamps) != len(features):
        continue  # Skip if mismatch
    
    # Predict with ensemble (average all folds)
    all_probs = []
    for model, scaler in zip(models, scalers):
        model.eval()
        feat_s = scaler.transform(features)
        with torch.no_grad():
            prob = torch.sigmoid(
                model(torch.tensor(feat_s, dtype=torch.float32))
            ).numpy().squeeze()
        all_probs.append(prob)
    
    probs = np.mean(all_probs, axis=0)
    
    # Ground truth segments
    gt_spans = bio_to_laugh_segments(df)
    if not gt_spans: continue
    
    # Merge predictions into segments
    pred_spans = merge_probs(probs, timestamps, 0.5)
    
    row = {'vid': vid, 'n_pred': len(pred_spans), 'n_gt': len(gt_spans)}
    for th in THRESHOLDS:
        p, r, f = segment_f1(pred_spans, gt_spans, th)
        row[f'p_{th}'] = round(p, 4)
        row[f'r_{th}'] = round(r, 4)
        row[f'f_{th}'] = round(f, 4)
        RESULTS[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
    per_video.append(row)

print(f'\nEvaluated: {len(per_video)}/{len(test_vids)} test videos')
print('='*65)
print('EMNLP TEST SET — IoU SEGMENT-LEVEL EVALUATION')
print(f'N = {len(per_video)} videos')
print('='*65)
print(f"{'IoU':>6} | {'Precision':>10} {'Recall':>10} {'F1':>10}")
print('-'*45)
for th in THRESHOLDS:
    rs = RESULTS[th]
    if not rs: continue
    pm = np.mean([x['p'] for x in rs])
    rm = np.mean([x['r'] for x in rs])
    fm = np.mean([x['f'] for x in rs])
    print(f' >={th:.1f} | {pm:.4f} {rm:.4f} {fm:.4f}')

print()
print('Per-video (IoU>=0.3):')
for row in sorted(per_video, key=lambda x: x.get('f_0.3', 0), reverse=True)[:10]:
    print(f"  {row['vid']:<18} gt={row['n_gt']:>3} pd={row['n_pred']:>4} F1={row.get('f_0.3', 0):.4f}")


In [ ]:
# Cell 8: Save Model + Results
# Save best model + CV results to Drive

# Save best model (highest fold F1)
best_idx = int(np.argmax(fold_f1s))
torch.save(models[best_idx].state_dict(), f'{WORK}/fusion_255_model.pt')
shutil.copy(f'{WORK}/fusion_255_model.pt', f'{BASE}/models/fusion_255_model.pt')
print(f'Model saved: {BASE}/models/fusion_255_model.pt')

# Save results
cv_results = {
    'n_train_val_videos': len(set(vids_list)),
    'n_test_videos': len(per_video),
    'positive_rate': float(pos_rate),
    'pos_weight': float(pos_weight),
    'cross_val_f1': float(np.mean(fold_f1s)),
    'cross_val_std': float(np.std(fold_f1s)),
    'fold_f1s': [float(f) for f in fold_f1s],
    'test_iou_summary': {},
}

for th in THRESHOLDS:
    rs = RESULTS[th]
    if rs:
        cv_results['test_iou_summary'][f'iou_{th}'] = {
            'f1': float(np.mean([x['f'] for x in rs])),
            'p': float(np.mean([x['p'] for x in rs])),
            'r': float(np.mean([x['r'] for x in rs])),
        }

with open(f'{WORK}/results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)
shutil.copy(f'{WORK}/results.json', f'{BASE}/models/fusion_255_results.json')
print(f'Results saved')
print(json.dumps(cv_results, indent=2))
